In [ ]:
import os, json, math, random, gc, time, ast
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, resnet18

from tqdm import tqdm

print("\u2705 Imports successful")
print(f"\u2705 CUDA available: {torch.cuda.is_available()}")

# BirdCLEF 2026 Inference v8 - 2-Model Ensemble (ResNet18 + EfficientNet-B0, TTA)
## Changes from v7 (nudges session):
- **Loads v8 model weights** (`resnet18_v8_fold_*.pt` / `efficientnet_v8_fold_*.pt`)
- **Test-Time Augmentation (TTA)**: averages predictions over 3 overlapping 5-second windows  
  (centre, −1 s offset, +1 s offset) — reduces variance without risking timeout
- **Uniform 0.5 thresholds** (unchanged from v7 — proven approach)

In [ ]:
# === SETUP ===
import os
from pathlib import Path

_candidate_dirs = [
    "/kaggle/input/birdclef-2026/test_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/test_soundscapes",
]
TEST_AUDIO_DIR = next((p for p in _candidate_dirs if os.path.isdir(p)), _candidate_dirs[0])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
    # v8 TTA: number of time-offset windows to average per segment
    # offsets in seconds: 0 (centre), -tta_offset_s, +tta_offset_s
    tta_offset_s=1,
)

print(f"Device: {device}")
print(f"Test audio dir: {TEST_AUDIO_DIR}")
if not os.path.isdir(TEST_AUDIO_DIR):
    print(f"WARNING: Test data dir not found! Tried: {_candidate_dirs}")
else:
    n_items = len(list(Path(TEST_AUDIO_DIR).iterdir()))
    print(f"Found test directory with {n_items} items")

In [ ]:
# === LOAD SPECIES & THRESHOLDS ===
species_paths = [
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v8-model/species.json",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v7-model/species.json",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/species.json",
    "species.json",
]

species = None
for path in species_paths:
    try:
        with open(path, "r") as f:
            species = json.load(f)
        print(f"\u2705 Loaded species from: {path}")
        break
    except FileNotFoundError:
        continue

if species is None:
    print("\u274c ERROR: Could not load species.json from any path!")
    raise FileNotFoundError("species.json not found")

n_classes = len(species)
print(f"\u2705 {n_classes} species loaded")

# Uniform 0.5 thresholds (unchanged from v7)
thresholds = {sp: 0.5 for sp in species}
print(f"\u2705 Uniform 0.5 thresholds set for {len(thresholds)} species")

In [ ]:
# === HELPER FUNCTIONS ===
def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr // 2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min = S_db.min()
    S_max = S_db.max()
    if S_max - S_min < 1e-9:
        S_norm = np.zeros_like(S_db, dtype=np.float32)
    else:
        S_norm = (S_db - S_min) / (S_max - S_min + 1e-9)
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)

print("\u2705 Helper functions defined")

In [ ]:
# === RESNET18 ARCHITECTURE (unchanged from v7) ===
class ResNet18Audio(nn.Module):
    """ResNet18-based classifier for mel spectrograms."""
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

print("\u2705 ResNet18Audio defined")

In [ ]:
# === EFFICIENTNET-B0 ARCHITECTURE (unchanged from v7) ===
class EfficientNetB0Audio(nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        self.model = efficientnet_b0(weights=None)
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        features = self.model(x)
        return self.head(features)

print("\u2705 EfficientNetB0Audio defined")

In [ ]:
# === LOAD ALL TRAINED MODELS (v8 weights) ===
print("Loading all 10 trained models (5 folds x 2 architectures)...")

MODEL_BASE_PATHS = [
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v8-model",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v7-model",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species",
]
MODEL_BASE = next((p for p in MODEL_BASE_PATHS if os.path.isdir(p)), MODEL_BASE_PATHS[0])
print(f"Loading models from: {MODEL_BASE}")

resnet_models = []
efficientnet_models = []

for fold_idx in range(5):
    # Load ResNet18 — try v8 first, fall back to v7
    for fname in [f"resnet18_v8_fold_{fold_idx}.pt", f"resnet18_v7_fold_{fold_idx}.pt"]:
        resnet_path = f"{MODEL_BASE}/{fname}"
        if os.path.exists(resnet_path):
            m = ResNet18Audio(n_classes=n_classes).to(device)
            m.load_state_dict(torch.load(resnet_path, map_location=device))
            m.eval()
            resnet_models.append(m)
            print(f"  \u2705 ResNet18 fold {fold_idx}: {fname}")
            break
    else:
        print(f"  \u26a0\ufe0f  ResNet18 fold {fold_idx}: not found — skipping")

    # Load EfficientNet-B0 — try v8 first, fall back to v7
    for fname in [f"efficientnet_v8_fold_{fold_idx}.pt", f"efficientnet_v7_fold_{fold_idx}.pt"]:
        eff_path = f"{MODEL_BASE}/{fname}"
        if os.path.exists(eff_path):
            m = EfficientNetB0Audio(n_classes=n_classes).to(device)
            m.load_state_dict(torch.load(eff_path, map_location=device))
            m.eval()
            efficientnet_models.append(m)
            print(f"  \u2705 EfficientNet fold {fold_idx}: {fname}")
            break
    else:
        print(f"  \u26a0\ufe0f  EfficientNet fold {fold_idx}: not found — skipping")

print(f"\n\u2705 Loaded {len(resnet_models)} ResNet18 + {len(efficientnet_models)} EfficientNet models")

In [ ]:
# === TEST AUDIO DATASET WITH TTA (v8 nudge) ===
# Each 5-second nominal window is predicted three times:
#   1. Centre window (standard)
#   2. Window shifted left  by tta_offset_s seconds
#   3. Window shifted right by tta_offset_s seconds
# The three probability vectors are averaged to reduce variance.

class TestAudioDataset(Dataset):
    """Yields one mel spectrogram per nominal 5-second window."""
    def __init__(self, audio_path, cfg):
        sr = cfg["sr"]
        seg_samples = sr * cfg["seconds"]

        try:
            y, sr0 = sf.read(audio_path, always_2d=False)
            if y.ndim == 2:
                y = y.mean(axis=1)
            if sr0 != sr:
                y = librosa.resample(y, orig_sr=sr0, target_sr=sr)
            y = y.astype(np.float32)
        except Exception as e:
            print(f"Warning: failed to load {audio_path}: {e}")
            y = np.zeros(seg_samples, dtype=np.float32)

        self.y = y
        self.cfg = cfg
        self.seg_samples = seg_samples
        self.n_windows = max(1, len(y) // seg_samples)

    def __len__(self):
        return self.n_windows

    def _extract_mel(self, start: int) -> torch.Tensor:
        """Extract and convert a 5-second slice starting at `start` samples."""
        end = start + self.seg_samples
        if end <= len(self.y):
            segment = self.y[start:end].copy()
        else:
            segment = np.zeros(self.seg_samples, dtype=np.float32)
            avail = len(self.y) - start
            if avail > 0:
                segment[:avail] = self.y[start:start + avail]
        mel = logmel_from_wave(segment, self.cfg["sr"])
        return torch.from_numpy(mel)[None, ...]  # (1, n_mels, T)

    def __getitem__(self, i):
        """Returns a tuple (centre_mel, left_mel, right_mel)."""
        centre = i * self.seg_samples
        offset = self.cfg["tta_offset_s"] * self.cfg["sr"]
        left  = max(0, centre - offset)
        right = min(len(self.y) - self.seg_samples, centre + offset)
        if right < 0:
            right = 0
        return (
            self._extract_mel(centre),
            self._extract_mel(left),
            self._extract_mel(right),
        )

print("\u2705 TestAudioDataset defined (TTA: 3 overlapping windows per segment)")

In [ ]:
# === PREDICTION FUNCTION WITH TTA (v8 nudge) ===
def get_predictions_for_audio(audio_path):
    """
    Get ensemble predictions with TTA.
    For each nominal 5-second window we predict on:
      - centre slice
      - slice shifted -tta_offset_s seconds
      - slice shifted +tta_offset_s seconds
    The 3 TTA predictions are averaged, then the model ensemble is averaged.
    Returns a list of numpy arrays, one per window.
    """
    dataset = TestAudioDataset(audio_path, CFG)
    loader  = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

    window_predictions = []

    with torch.no_grad():
        for centre_x, left_x, right_x in loader:
            # TTA inputs: shape (1, 1, n_mels, T) each
            tta_inputs = [
                centre_x.to(device),
                left_x.to(device),
                right_x.to(device),
            ]
            window_preds = np.zeros(n_classes, dtype=np.float32)
            n_preds = 0

            for x_tta in tta_inputs:
                # ResNet18 ensemble
                for model in resnet_models:
                    probs = torch.sigmoid(model(x_tta)).cpu().numpy()[0]
                    window_preds += probs
                    n_preds += 1
                # EfficientNet ensemble
                for model in efficientnet_models:
                    probs = torch.sigmoid(model(x_tta)).cpu().numpy()[0]
                    window_preds += probs
                    n_preds += 1

            if n_preds > 0:
                window_preds /= n_preds
            window_predictions.append(window_preds)

    return window_predictions

print("\u2705 Prediction function defined (TTA x3 + 10-model ensemble)")

In [ ]:
# === GENERATE PREDICTIONS ===
test_dir = Path(TEST_AUDIO_DIR)

test_files = []
if test_dir.exists():
    for pattern in ["*.ogg", "*.wav", "*.mp3", "*.flac", "*.m4a"]:
        test_files.extend(test_dir.rglob(pattern))
    test_files = sorted(set(test_files))

print(f"Found {len(test_files)} test files in {TEST_AUDIO_DIR}")
if test_files:
    from collections import Counter
    fmts = Counter(f.suffix.lower() for f in test_files)
    for fmt, count in sorted(fmts.items()):
        print(f"  {fmt}: {count} files")
else:
    print("WARNING: No test audio files found!")

predictions = {}

for audio_path in tqdm(test_files, desc="Predicting"):
    stem = audio_path.stem
    window_preds = get_predictions_for_audio(str(audio_path))

    sr        = CFG["sr"]
    seg_s     = CFG["seconds"]
    for w_idx, preds in enumerate(window_preds):
        end_s    = (w_idx + 1) * seg_s
        row_id   = f"{stem}_{end_s}"
        predictions[row_id] = preds

print(f"\n\u2705 Generated predictions for {len(predictions)} windows")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# === CHECK TEST METADATA ===
print("Checking test metadata...")
test_meta = None
try:
    test_meta = pd.read_csv("/kaggle/input/competitions/birdclef-2026/test.csv")
    print(f"\u2705 Test metadata shape: {test_meta.shape}")
    print(f"Test metadata columns: {test_meta.columns.tolist()}")
    print(f"\nFirst 10 rows:")
    print(test_meta.head(10))
except FileNotFoundError:
    print("\u26a0\ufe0f  No test.csv found - checking alternative paths...")
    import glob
    csv_files = glob.glob("/kaggle/input/competitions/birdclef-2026/*.csv")
    print(f"Available CSV files: {csv_files}")

In [ ]:
# === CREATE SUBMISSION (Uniform 0.5 Thresholds) ===
sample_sub_paths = [
    "/kaggle/input/birdclef-2026/sample_submission.csv",
    "/kaggle/input/competitions/birdclef-2026/sample_submission.csv",
]
sample_sub = None
for path in sample_sub_paths:
    try:
        sample_sub = pd.read_csv(path)
        print(f"Loaded sample_submission.csv from: {path}")
        print(f"Shape: {sample_sub.shape}")
        break
    except FileNotFoundError:
        continue

if sample_sub is None:
    raise FileNotFoundError("sample_submission.csv not found!")

species_cols = [col for col in sample_sub.columns if col != "row_id"]
print(f"Species columns: {len(species_cols)}")

# Fill submission with predictions
submission_df = sample_sub.copy()

for col in species_cols:
    submission_df[col] = submission_df[col].astype(float)

sp_to_idx = {sp: i for i, sp in enumerate(species)}

filled_rows  = 0
missing_rows = 0

for i, row_id in enumerate(submission_df["row_id"]):
    if row_id in predictions:
        preds = predictions[row_id]
        for col in species_cols:
            sp_idx = sp_to_idx.get(col)
            if sp_idx is not None:
                submission_df.at[i, col] = float(preds[sp_idx])
            else:
                submission_df.at[i, col] = 0.0   # species not in training taxonomy
        filled_rows += 1
    else:
        for col in species_cols:
            submission_df.at[i, col] = 0.0
        missing_rows += 1

print(f"\u2705 Filled {filled_rows} rows with predictions")
print(f"\u26a0\ufe0f  {missing_rows} rows not found in predictions (set to 0.0)")
print(f"\nFirst 5 rows of submission:")
print(submission_df.head())

In [ ]:
# === SAVE SUBMISSION ===
submission_df.to_csv("/kaggle/working/submission.csv", index=False)

print(f"\u2705 Submission saved to: /kaggle/working/submission.csv")
print(f"\n\U0001f4ca SUBMISSION SUMMARY:")
print(f"  Total predictions: {len(submission_df)}")
print(f"  Total species: {len(species_cols)}")
if len(submission_df) > 0:
    print(f"  Avg probability per species: {submission_df[species_cols].mean().mean():.4f}")
print(f"\n\u2705 INFERENCE COMPLETE!")
print(f"\n\U0001f4c8 ENSEMBLE STRATEGY (v8):")
print(f"  - 5 ResNet18 folds (v8 weights — trained with mixup + grad-clip + AUC selection)")
print(f"  - 5 EfficientNet-B0 folds")
print(f"  - TTA: 3 overlapping windows per segment (centre, -{CFG['tta_offset_s']}s, +{CFG['tta_offset_s']}s)")
print(f"  - Dynamic windowing (one row per 5-second segment from actual audio duration)")
print(f"  - Uniform 0.5 thresholds")

In [ ]:
# === VALIDATE SUBMISSION FORMAT ===
print("Validating submission format...")

nan_count = submission_df.isna().sum().sum()
if nan_count > 0:
    print(f"\u26a0\ufe0f  WARNING: Found {nan_count} NaN values in submission!")
else:
    print("\u2705 No NaN values")

wrong_dtypes = []
for col in submission_df.columns:
    expected = "object" if col == "row_id" else "float64"
    if str(submission_df[col].dtype) != expected:
        wrong_dtypes.append((col, submission_df[col].dtype, expected))

if wrong_dtypes:
    print(f"\u26a0\ufe0f  WARNING: {len(wrong_dtypes)} columns have wrong dtype!")
    for col, actual, expected in wrong_dtypes[:5]:
        print(f"  {col}: {actual} (expected {expected})")
else:
    print("\u2705 All dtypes correct")

out_of_range = ((submission_df[species_cols] < 0.0) | (submission_df[species_cols] > 1.0)).sum().sum()
if out_of_range > 0:
    print(f"\u26a0\ufe0f  WARNING: {out_of_range} values outside [0, 1]!")
else:
    print("\u2705 All values in [0, 1]")

print(f"\n\u2705 Validation complete")